# T303c + T303e — Train v2 with CoSENT loss on graded labels

**Hypothesis:** feeding the model graded labels (0.0 / 0.4 / 0.5 / 0.85 / 1.0) via CoSENT loss produces better ranking than the v1.2 MNR run that threw away everything except label==1.0.

**Baseline to beat (v1.2, dev-149):**
- graded nDCG@10: **0.554**
- graded Recall@5: **0.485**
- MRR (binary): 0.468

**Bar for promotion:** v2 must beat v1.2 by ≥10% on graded nDCG@10 (target ≥0.610) to replace v1.2 as the production model. Per Phase 3 plan's budget guardrail.

**Why CoSENT:**
- MNR only learns from positive pairs; ignores our 8,361 graded + negative pairs.
- CoSENT learns a pairwise ranking: it just needs pairs with relative labels. It uses ALL 26,760 pairs.
- No in-batch negative sampling, so batch size can go higher (32) without the brittle MNR dynamics.

**Runtime:** ~35-50 min on Colab Free T4.

**Before running:**
- Runtime → Change runtime type → **T4 GPU**
- HF token with write scope ready

## 1. Install + auth

In [ ]:
!pip install -q sentence-transformers==3.4.1 datasets==3.2.0 huggingface_hub==0.27.0

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Pull training pairs — use ALL labels this time

In [ ]:
from datasets import load_dataset
from collections import Counter

PAIRS_REPO = 'ManmohanBuildsProducts/auto-parts-search-pairs'
ds = load_dataset(PAIRS_REPO, split='train')
print(f'loaded {len(ds)} pairs')

print('label distribution:')
for lbl, n in sorted(Counter(f"{r['label']:.2f}" for r in ds).items()):
    print(f'  {lbl}: {n}')

## 3. Build InputExamples with graded labels

In [ ]:
import random
from sentence_transformers import InputExample

SEED = 42
random.seed(SEED)

examples = [
    InputExample(texts=[r['text_a'], r['text_b']], label=float(r['label']))
    for r in ds
]
random.shuffle(examples)
print(f'{len(examples)} training examples (all labels kept)')

## 4. Load BGE-m3 + CoSENT loss

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

BASE = 'BAAI/bge-m3'
print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

model = SentenceTransformer(BASE)
model.max_seq_length = 128

BATCH = 32
loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
train_loss = losses.CoSENTLoss(model)

EPOCHS = 1
WARMUP = int(len(loader) * 0.1)
print(f'steps/epoch: {len(loader)}, warmup: {WARMUP}')

## 5. Train

In [ ]:
OUT_DIR = 'auto-parts-search-v2'

model.fit(
    train_objectives=[(loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={'lr': 2e-5},
    output_path=OUT_DIR,
    show_progress_bar=True,
    use_amp=True,
)
print('training done.')

## 6. Push to HF Hub (private)

In [ ]:
from huggingface_hub import HfApi

MODEL_REPO = 'ManmohanBuildsProducts/auto-parts-search-v2'
api = HfApi()
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=MODEL_REPO,
    commit_message=f'v2 candidate — BGE-m3 fine-tuned on {len(examples)} graded pairs, 1 epoch, CoSENT loss',
)
print(f'pushed to https://huggingface.co/{MODEL_REPO}')

## 7. Sanity check

In [ ]:
probes = ['patti badal do', 'brake pad Maruti Swift', 'kick starter', 'engine noise abnormal']
emb = model.encode(probes, normalize_embeddings=True)
print('embedding shape:', emb.shape)

## Next (locally)

```bash
python3.11 -m training.evaluate_graded \
    --model ManmohanBuildsProducts/auto-parts-search-v2 \
    --graded data/training/experiments/2026-04-13-graded/benchmark_dev_graded.jsonl \
    --out data/training/experiments/2026-04-13-graded/v2-graded.json
```

Decision gate: v2 graded nDCG@10 must beat v1.2 (0.554) by ≥10% (target ≥0.610).
- ✅ → promote v2 to production; write ADR 013 (graded labels win) + ADR 014 (CoSENT loss win).
- ❌ → keep v1.2 as production; ADR 013 records that binary + MNR was sufficient at our data scale.